# Inference API Tests

Test the three implemented endpoints of the KudiScore credit scoring service:
- `GET /health` — Server status and model readiness
- `POST /predict/score` — Get credit scores and default probabilities
- `POST /predict/explain` — SHAP-based feature importance explanations

## Setup

Import dependencies and configure the API base URL.

In [7]:
import requests
import json
import pandas as pd
from datetime import datetime

BASE_URL = "http://localhost:8000"
print(f"Testing KudiScore endpoints at {BASE_URL}")
print("=" * 80)

Testing KudiScore endpoints at http://localhost:8000


## 1. Health Check

Verify the server is running and the credit model is initialized correctly.

In [8]:
print("\n[1] GET /health")
print("-" * 80)
try:
    resp = requests.get(f"{BASE_URL}/health", timeout=5)
    print(f"Status: {resp.status_code}")
    health = resp.json()
    print(f"Response:\n{json.dumps(health, indent=2)}")
except Exception as e:
    print(f"❌ Error: {e}")


[1] GET /health
--------------------------------------------------------------------------------
Status: 200
Response:
{
  "status": "ok",
  "models_loaded": [
    "credit"
  ],
  "versions": {
    "credit": "v1.0"
  },
  "metrics": {
    "roc_auc_uncalibrated_test": 0.768,
    "roc_auc_calibrated_test": 0.7637917659142064,
    "pr_auc_test": 0.2006595607836806,
    "brier_test": 0.07593898710428353,
    "logloss_test": 0.2634689063277627,
    "ks_test": 0.4420139420030157,
    "approval_rate_at_5pct_pd": 0.458,
    "default_rate_among_approved": 0.017467248908296942
  }
}


## 2. Score Prediction

Send a user's features to get their credit score (300-850) and default probability (0-1).

In [12]:
print("\n[2] POST /predict/score")
print("-" * 80)

# Load a test user from the feature matrix
features_df = pd.read_parquet("../data/features_v1.parquet")
test_user_row = features_df.iloc[4]
user_id = str(test_user_row['user_id'])

# Build the request payload
features_dict = test_user_row.drop('user_id').to_dict()

score_request = {
    "user_id": user_id,
    "features": {k: v if pd.notna(v) else None for k, v in features_dict.items()}
}

print(f"Test user: {user_id}")
print(f"Features sent: {len(score_request['features'])} features")

try:
    resp = requests.post(
        f"{BASE_URL}/predict/score",
        json=score_request,
        timeout=10
    )
    print(f"Status: {resp.status_code}")
    if resp.status_code == 200:
        result = resp.json()
        print(f"\n✓ Prediction successful:")
        print(f"  Score:         {result['score']}")
        print(f"  PD:            {result['pd']:.4f}")
        print(f"  Model version: {result['model_version']}")
        print(f"  Computed at:   {result['computed_at']}")
        print(f"  Sub-scores:    {json.dumps(result['sub_scores'], indent=4)}")
    else:
        print(f"❌ Error: {resp.json()}")
except Exception as e:
    print(f"❌ Exception: {e}")


[2] POST /predict/score
--------------------------------------------------------------------------------
Test user: e4e08096-c520-4423-b8ee-8a1e808a9a03
Features sent: 42 features
Status: 200

✓ Prediction successful:
  Score:         761
  PD:            0.0652
  Model version: v1.0
  Computed at:   2026-05-11T12:49:17.442033+00:00
  Sub-scores:    {
    "cash_flow_stability": 50,
    "customer_base": 49,
    "growth": 50,
    "reliability": 51
}


## 3. Explain Prediction

Get SHAP-based explanations: which features help (lower PD) or hurt (raise PD) the credit score.

In [10]:
print("\n[3] POST /predict/explain")
print("-" * 80)

# Use the same user from the score prediction
explain_request = {
    "user_id": user_id,
    "features": {k: v if pd.notna(v) else None for k, v in features_dict.items()}
}

print(f"Test user: {user_id}")

try:
    resp = requests.post(
        f"{BASE_URL}/predict/explain",
        json=explain_request,
        timeout=15
    )
    print(f"Status: {resp.status_code}")
    if resp.status_code == 200:
        result = resp.json()
        print(f"\n✓ Explanation successful:")
        print(f"  Score: {result['score']}")
        print(f"  PD:    {result['pd']:.4f}")
        print(f"  Model: {result['model_version']}")
        
        print(f"\n  📈 Factors HELPING the score (lowering PD):")
        for i, factor in enumerate(result['helping'], 1):
            print(f"    {i}. {factor['feature']}")
            print(f"       Value: {factor['value']}")
            print(f"       Reason: {factor['phrasing']}")
            print(f"       Impact: +{factor['score_delta']} points\n")
        
        print(f"  📉 Factors HURTING the score (raising PD):")
        for i, factor in enumerate(result['hurting'], 1):
            print(f"    {i}. {factor['feature']}")
            print(f"       Value: {factor['value']}")
            print(f"       Reason: {factor['phrasing']}")
            print(f"       Impact: {factor['score_delta']} points\n")
    else:
        print(f"❌ Error: {resp.json()}")
except Exception as e:
    print(f"❌ Exception: {e}")


[3] POST /predict/explain
--------------------------------------------------------------------------------
Test user: 9a92cf82-dca0-4339-a3ad-89399e17335b
Status: 200

✓ Explanation successful:
  Score: 850
  PD:    0.0179
  Model: v1.0

  📈 Factors HELPING the score (lowering PD):
    1. txns_per_active_day_30d
       Value: 10.3
       Reason: txns_per_active_day_30d = 10.3
       Impact: +3 points

    2. std_intertxn_seconds_30d
       Value: 16326.738156199772
       Reason: std_intertxn_seconds_30d = 16326.738156199772
       Impact: +3 points

  📉 Factors HURTING the score (raising PD):


## Summary

Tested all 3 implemented endpoints for KudiScore credit scoring service.

**To run these tests:**
```bash
# Terminal 1: Start the FastAPI server
cd /Users/mac/Projects/Portfolio/Trace/ml_service
uvicorn app:app --port 8000 --reload

# Terminal 2: Run this notebook (execute each cell top-to-bottom)
```

The server loads the trained LightGBM model and isotonic calibrator from `models/model_artifact_v1.pkl` and provides:
- **Scoring**: Calibrated default probability (PD) with 4 sub-score components
- **Explainability**: SHAP-based feature importance identifying top-5 positive/negative factors